# Venom API / KPI Tutorial

这个 notebook 是 Venom 的端到端使用说明。这里的 **KPI** 按项目使用语境理解为：训练能否跑通、checkpoint 是否保存、采样是否能出图、不同模型族的核心 API 是否一致、guidance/conditioning 是否能按预期调用。

Venom 现在覆盖六类生成模型：

- Diffusion / Score SDE / Flow Matching / One-step generation: `venom.diffusion`
- VAE family: `venom.vae`
- Normalizing flows: `venom.flows`
- GAN family: `venom.gan`
- Energy-based models: `venom.ebm`
- Shared MNIST data and utilities: `venom.data`, `venom.utils`

所有示例默认用 MNIST，训练产物保存在 `runs/` 下。为了快速验证，下面多数命令使用 `--epochs 1`；正式训练时可以改成 `5`、`20` 或更大。

## 0. 安装与环境检查

在项目根目录运行这个 notebook。第一次使用建议先安装 editable package。

In [ ]:
# 如果你还没安装 Venom，先取消下一行注释执行。
# %pip install -e .

import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Current working directory:", Path.cwd())
print("Python:", sys.version)

In [ ]:
import torch
import venom

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("venom top-level exports loaded:", len(venom.__all__))

## 1. 总体 KPI 检查表

每个模型族都按同一套最小 KPI 验证：

| KPI | 你应该看到什么 |
|---|---|
| 训练入口 | 对应 `train_*.py` 能启动，loss/bpd 能打印。 |
| checkpoint | `runs/<family>/<variant>/model_001.pt` 被保存。 |
| preview grid | `samples_001.png` 被保存。 |
| 采样入口 | 对应 `sample_*.py` 能读取 checkpoint 并保存图片。 |
| Python API | `build_mnist_*` 或模型类能直接构建并调用 `training_loss` / `sample` / `log_prob`。 |
| guidance / conditioning | diffusion 的 classifier guidance / CFG，VAE/GAN/EBM 的 label conditioning 能通过 `--labels` 或 `y` 调用。 |

## 2. Diffusion / Score / Flow Matching / One-step Models

这一类模型都在 `venom.diffusion`。训练入口是 `train_diffusion.py`，采样入口是 `sample_diffusion.py`。

支持的核心 variant：

- Discrete diffusion: `ddpm`, `improved-ddpm`, `adm`, `cfg`
- Continuous score/SDE: `ncsn`, `ncsnv2`, `score-sde-vp`, `score-sde-ve`, `score-sde-subvp`
- EDM/PFGM: `edm`, `pfgm`, `pfgm++`
- Flow matching family: `rectified-flow`, `flow-matching`, `conditional-flow-matching`, `ot-cfm`, `stochastic-interpolants`
- One-step/few-step family: `consistency`, `shortcut`, `meanflow`
- Backbones: `--backbone unet` or `--backbone dit`

In [ ]:
# Fast smoke test: train DDPM for 1 epoch.
!python train_diffusion.py --variant ddpm --epochs 1 --batch-size 128 --sample-steps 20

In [ ]:
# Sample from a DDPM-family checkpoint with DDIM.
!python sample_diffusion.py \
  --checkpoint runs/mnist_diffusion/ddpm/model_001.pt \
  --sampler ddim \
  --sample-steps 20 \
  --num-samples 64 \
  --out runs/mnist_diffusion/ddpm/notebook_ddim_samples.png

### Diffusion guidance

Venom 里 diffusion guidance 主要分三种：

1. **Class conditioning**: `adm` 训练时直接使用类别标签。
2. **Classifier-free guidance**: `cfg` 用 `--class-dropout` 训练，采样时传 `--guidance-scale`。
3. **Classifier guidance**: 先训练 noised classifier，再采样时传 `--classifier-checkpoint` 和 `--classifier-scale`。

In [ ]:
# Train classifier-free guidance model.
!python train_diffusion.py --variant cfg --epochs 1 --class-dropout 0.1 --sample-steps 20

In [ ]:
# Sample with classifier-free guidance.
!python sample_diffusion.py \
  --checkpoint runs/mnist_diffusion/cfg/model_001.pt \
  --sampler dpm-solver++ \
  --sample-steps 20 \
  --labels 0,1,2,3,4,5,6,7,8,9 \
  --guidance-scale 3.0 \
  --out runs/mnist_diffusion/cfg/notebook_cfg_samples.png

In [ ]:
# Classifier guidance path: train classifier, then sample ADM/DDPM-family checkpoint with it.
# 这两个命令会稍慢；需要时再运行。
# !python -m venom.diffusion.train_classifier_mnist --epochs 1
# !python sample_diffusion.py \
#   --checkpoint runs/mnist_diffusion/adm/model_001.pt \
#   --sampler ddpm \
#   --labels 0,1,2,3,4,5,6,7,8,9 \
#   --classifier-checkpoint runs/mnist_diffusion/classifier/classifier_001.pt \
#   --classifier-scale 1.0

In [ ]:
# Python API: build and call training_loss / sampler directly.
import torch
from venom.diffusion.factory import build_mnist_diffusion
from venom.diffusion.samplers import make_sampler

model, diffusion = build_mnist_diffusion("ddpm", timesteps=1000, backbone="unet")
images = torch.randn(4, 1, 28, 28)
loss = diffusion.training_loss(images)
sampler = make_sampler(diffusion, "ddim", steps=10)
print("loss:", float(loss.detach()))

## 3. VAE Family

VAE 模型在 `venom.vae`，训练入口是 `train_vae.py`，采样入口是 `sample_vae.py`。

支持：`vae`, `conv-vae`, `beta-vae`, `cvae`, `iwae`, `vq-vae`, `ladder-vae`, `hierarchical-vae`, `flow-vae`。

其中 `cvae` 支持 label conditioning，采样时可以传 `--labels`。

In [ ]:
!python train_vae.py --variant conv-vae --epochs 1 --batch-size 128

In [ ]:
!python sample_vae.py \
  --checkpoint runs/mnist_vae/conv-vae/model_001.pt \
  --num-samples 64 \
  --out runs/mnist_vae/conv-vae/notebook_samples.png

In [ ]:
# Conditional VAE example.
# !python train_vae.py --variant cvae --epochs 1
# !python sample_vae.py \
#   --checkpoint runs/mnist_vae/cvae/model_001.pt \
#   --labels 0,1,2,3,4,5,6,7,8,9

In [ ]:
from venom.vae.factory import build_mnist_vae

vae = build_mnist_vae("conv-vae", latent_dim=64)
images = torch.randn(4, 1, 28, 28)
loss = vae.training_loss(images)
samples = vae.sample(batch_size=4, device=images.device)
print("vae loss:", float(loss.detach()), "samples:", tuple(samples.shape))

## 4. Normalizing Flows

Normalizing flows 在 `venom.flows`，区别于 `venom.diffusion.flow_matching`。这里的 flow 是可逆变换 + change-of-variables likelihood。

支持：`planar-flow`, `radial-flow`, `nice`, `realnvp`, `glow`, `maf`, `iaf`, `neural-spline-flow`, `ffjord`, `flow++`。

核心 KPI 是 bits-per-dim 能下降，并且 `sample_flow.py` 能从 Gaussian latent inverse 到图片空间。

In [ ]:
!python train_flow.py --variant realnvp --epochs 1 --batch-size 128 --num-layers 4 --hidden-dim 256

In [ ]:
!python sample_flow.py \
  --checkpoint runs/mnist_flow/realnvp/model_001.pt \
  --num-samples 64 \
  --out runs/mnist_flow/realnvp/notebook_samples.png

In [ ]:
from venom.flows import build_mnist_flow

flow, config = build_mnist_flow("realnvp", hidden_dim=256, num_layers=4)
images = torch.randn(4, 1, 28, 28)
bpd = flow.training_loss(images)
samples = flow.sample(batch_size=4, device=images.device)
print(config)
print("bits/dim:", float(bpd.detach()), "samples:", tuple(samples.shape))

## 5. GAN Family

GAN 模型在 `venom.gan`，训练入口是 `train_gan.py`，采样入口是 `sample_gan.py`。

支持：`gan`, `dcgan`, `cgan`, `acgan`, `infogan`, `lsgan`, `wgan`, `wgan-gp`, `hinge-gan`, `sn-gan`。

Conditioning：`cgan` 和 `acgan` 支持标签；采样时可以传 `--labels`。

In [ ]:
!python train_gan.py --variant dcgan --epochs 1 --batch-size 128

In [ ]:
!python sample_gan.py \
  --checkpoint runs/mnist_gan/dcgan/model_001.pt \
  --num-samples 64 \
  --out runs/mnist_gan/dcgan/notebook_samples.png

In [ ]:
# Conditional GAN example.
# !python train_gan.py --variant cgan --epochs 1
# !python sample_gan.py \
#   --checkpoint runs/mnist_gan/cgan/model_001.pt \
#   --labels 0,1,2,3,4,5,6,7,8,9

In [ ]:
from venom.gan import build_mnist_gan

generator, discriminator, gan_config = build_mnist_gan("dcgan", latent_dim=128)
z = torch.randn(4, 128)
fake = generator(z)
disc_out = discriminator(fake)
print(gan_config)
print("fake:", tuple(fake.shape), "logits:", tuple(disc_out["logits"].shape))

## 6. Energy-Based Models

EBM 模型在 `venom.ebm`，训练入口是 `train_ebm.py`，采样入口是 `sample_ebm.py`。

支持：`rbm`, `gaussian-rbm`, `conditional-rbm`, `conv-rbm`, `deep-ebm`, `conditional-ebm`, `jem`, `score-matching-ebm`, `sliced-score-matching-ebm`, `nce-ebm`。

采样方式：RBM-family 用 Gibbs；deep EBM/JEM 用 Langevin/SGLD。`conditional-rbm` 和 `conditional-ebm` 支持标签。

In [ ]:
!python train_ebm.py --variant rbm --epochs 1 --batch-size 128 --cd-steps 1

In [ ]:
!python sample_ebm.py \
  --checkpoint runs/mnist_ebm/rbm/model_001.pt \
  --steps 100 \
  --num-samples 64 \
  --out runs/mnist_ebm/rbm/notebook_samples.png

In [ ]:
from venom.ebm import RBM, cd_loss

rbm = RBM(hidden_dim=128)
images = torch.rand(4, 1, 28, 28)
loss, negatives = cd_loss(rbm, images, steps=1)
print("cd loss:", float(loss.detach()), "negatives:", tuple(negatives.shape))

## 7. 推荐的 smoke-test 顺序

如果你只想确认整个包都能跑，建议按这个顺序：

1. `python train_vae.py --variant conv-vae --epochs 1`
2. `python train_gan.py --variant dcgan --epochs 1`
3. `python train_flow.py --variant realnvp --epochs 1 --num-layers 4 --hidden-dim 256`
4. `python train_ebm.py --variant rbm --epochs 1`
5. `python train_diffusion.py --variant ddpm --epochs 1 --sample-steps 20`
6. `python train_diffusion.py --variant cfg --epochs 1 --class-dropout 0.1 --sample-steps 20`

这六步覆盖 reconstruction、adversarial training、exact likelihood、energy training、diffusion sampling 和 CFG guidance。

## 8. 产物位置

| 模型族 | Checkpoint 目录 | 采样脚本 |
|---|---|---|
| Diffusion / Score / Flow Matching / One-step | `runs/mnist_diffusion/<variant>/` | `sample_diffusion.py` |
| VAE | `runs/mnist_vae/<variant>/` | `sample_vae.py` |
| Normalizing Flow | `runs/mnist_flow/<variant>/` | `sample_flow.py` |
| GAN | `runs/mnist_gan/<variant>/` | `sample_gan.py` |
| EBM | `runs/mnist_ebm/<variant>/` | `sample_ebm.py` |

每个训练脚本通常保存：

- `model_001.pt`, `model_002.pt`, ...
- `samples_001.png`, `samples_002.png`, ...

这就是最基本的项目 KPI：能训练、能保存、能采样、能复现调用路径。